# 05 — Daily signal demo
Runs the incremental daily pipeline: incremental price/macro/news download, feature rebuild, model scoring. Writes `reports/latest_trades.csv` with high-conviction signals (p≥0.55) plus a top-10 watchlist.

In [3]:
# Kernel bootstrap
import sys, os
def _find_root():
    for base in [os.getcwd(), os.path.abspath(".."), os.path.abspath(".")]:
        for cand in [base, os.path.dirname(base)]:
            if os.path.isfile(os.path.join(cand, "requirements.txt")) and os.path.isdir(os.path.join(cand, "src")):
                return cand
    return os.path.abspath("..")
_project_root = _find_root()
if _project_root not in sys.path: sys.path.insert(0, _project_root)
os.chdir(_project_root)
from src.bootstrap import ensure
ensure({"xgboost": "xgboost", "optuna": "optuna"})
print("bootstrap OK  project_root =", _project_root)

bootstrap OK  project_root = /home/yazadanparast/Downloads/eu-midcap-echo


In [4]:
%load_ext autoreload
%autoreload 2
import subprocess, sys
print('running daily_predict.py ( incremental download+predict)...')
r = subprocess.run([sys.executable, 'scripts/daily_predict.py'],
                   capture_output=True, text=True, cwd=_project_root)
print(r.stdout[-2000:])
if r.returncode != 0:
    print('STDERR (tail):'); print(r.stderr[-1500:])


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
running daily_predict.py (takes ~1 minute for incremental download+predict)...
[telegram] credentials not set; skipping.



In [5]:
%load_ext autoreload
%autoreload 2
import pandas as pd, os
f = os.path.join(_project_root, 'reports/latest_trades.csv')
if os.path.exists(f):
    trades = pd.read_csv(f)
    print('latest trades:', trades.shape)
    # signals vs watchlist
    if 'type' in trades.columns:
        sig = trades[trades['type']=='signal']
        wl  = trades[trades['type']=='watchlist']
        print(f'  high-conviction signals (p>=0.55): {len(sig)}')
        print(f'  top-10 watchlist: {len(wl)}')
    pd.set_option('display.max_columns', None)
    trades
else:
    print('no latest_trades.csv yet.')


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
latest trades: (10, 10)
  high-conviction signals (p>=0.55): 3
  top-10 watchlist: 7
